# 第25章 最新LLMの部品を移植する

RoPE・RMSNorm・SwiGLU・MoEを自分のGPTに入れて、公平に比べる

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/25-modern.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：第12章のGPTを、LlamaやQwenと同じ部品（RMSNorm・SwiGLU・RoPE）に置き換えた「現代版ミニGPT」と、その効果を確かめる比較実験。MoE層の試作も
- **身につく数学・理論**：RMSNorm（LayerNormの簡略版） ／ SwiGLU（門番つきのフィードフォワード） ／ RoPE（回転位置埋め込み）と相対位置 ／ 公平な比較実験の作り方（同じ条件・複数のシード） ／ MoE（専門家の混合）とルーティング ／ 長いコンテキスト・推論モデル・状態空間モデル・マルチモーダル・解釈可能性の地図
- **新しく使うPython・PyTorch**：`torch.rsqrt` ／ `F.silu` ／ 三角関数 `torch.cos` / `torch.sin` ／ `F.scaled_dot_product_attention`（第16章）

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（CPUでOK） |
| 所要時間目安 | 2〜3時間 |
| 難易度 | ★★★★★ |

> **✅ CPUでそのまま実行できます**：上のセルから順に再生ボタン（▶）または `Shift + Enter` で実行してください。

> **🎯 この章でつくるもの**
>
> 第12章のGPTは、2019年のGPT-2とほぼ同じ設計でした。その後のLLM（Llama、Qwen、Gemmaなど）は、いくつかの部品を改良しています。この章では、その代表的な部品を**自分のGPTに移植**し、本当に良くなるのかを**公平な実験**で確かめます。論文を読んで「良いらしい」で終わらせず、自分の手で確かめる。それが研究の第一歩です。

> **🔗 これまでの復習とつながり：第10章・第11章・第12章とのつながり**
>
> この章で移植する4つの最新部品は、どれも第2〜3部で自分の手で書いた部品の「進化版」です！
>
> - **第10章（LayerNorm）** → 平均引きを省いて速くシンプルにした **RMSNorm** へ
> - **第11章（Attention と足し算式の位置埋め込み `pos_emb`）** → QueryとKeyのベクトルを2D回転させて相対位置 $m - n$ を自然に伝える **RoPE** へ
> - **第12章（FeedForward と Transformer Block）** → 門番つきの **SwiGLU**、さらに専門家を使い分ける **MoE** へ

**ミナト**：最新のモデルの部品って、難しそうです……。

**ボンゴ先生**：1つ1つは、驚くほど短いコードだよ。RMSNormは5行、SwiGLUは3行、RoPEも10行ほど。大事なのは部品の難しさじゃなくて、「本当に良くなったのか？」を確かめる**実験の作法**のほうだ。今日はキミが研究者だ。

## 準備と、比べる相手（ベースライン）

> **✨ コツ：実行環境とリソースの目安**
>
> - **環境**：CPUで完走できます（GPU不要）。
> - **ダウンロードとメモリ**：初回に `minigpt.py`（約4KB）と青空文庫『こころ』（約350KB）を取得します。必要RAMは約1GBです。
> - **所要時間**：1500ステップの学習1回あたりCPUで2〜5分（ベースライン2シード＋現代版2シードの計4回で合計10〜20分、GPUなら合計1分未満）。時間がないときは `steps=500`・`seed in [0]` で動作確認してから本実験へ進めます。推奨ライブラリは `requirements-book.txt` を参照してください。

第12章と同じ小さな設定（2層・幅64・窓64・1500ステップ）を使います。CPUでも1回数分で終わるので、何度も実験できます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
!wget -nv https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/minigpt.py
import os, re, math, urllib.request
if not os.path.exists("minigpt.py"):
    raise RuntimeError("minigpt.py のダウンロードに失敗しました。ネットワーク接続を確認して再実行するか、付録E（エラー辞典）を参照してください。")
import torch
import torch.nn as nn
import torch.nn.functional as F
from minigpt import GPT

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/"
           "0984f7dc18a2270bdfd2ff99bf14fd1b3ebb4f65/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    try:
        text = urllib.request.urlopen(url, timeout=30).read().decode("cp932")
    except Exception as e:
        raise RuntimeError(
            f"青空文庫の取得に失敗しました（{e}）。ネットワーク接続を確認して再実行するか、付録E（エラー辞典）を参照してください。"
        ) from e
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text])
n = int(len(data) * 0.9)
train_data, val_data = data[:n], data[n:]
T = 64

def get_batch(data, g):
    ix = torch.randint(0, len(data) - T, (32,), generator=g)
    return torch.stack([data[i:i + T] for i in ix]), torch.stack([data[i + 1:i + T + 1] for i in ix])

@torch.no_grad()
def val_loss(model):
    model.eval()
    starts = range(0, len(val_data) - T - 1, T)
    xs = torch.stack([val_data[i:i + T] for i in starts])
    ys = torch.stack([val_data[i + 1:i + T + 1] for i in starts])
    loss = model(xs, ys)[1].item()
    model.train()
    return loss

def experiment(make_model, seed, steps=1500):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = make_model()
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    for step in range(steps):
        x, y = get_batch(train_data, g)
        _, loss = model(x, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return val_loss(model), sum(p.numel() for p in model.parameters())

base = lambda: GPT(V, block_size=T, n_embd=64, n_head=4, n_layer=2, dropout=0.0)   # 現代版に合わせてドロップアウトなし
results = {"ベースライン（GPT-2型）": [experiment(base, seed) for seed in [0, 1]]}
for name, rs in results.items():
    print(name, "検証損失：", [round(r[0], 3) for r in rs], " パラメータ：", rs[0][1])

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ベースライン（GPT-2型） 検証損失： [3.21, 3.275]  パラメータ： 369677
```

</details>

> **💡 理論のポイント：公平な比較の作法**
>
> 「新しい部品で良くなった」と言うためには、**部品以外のすべてを同じにする**必要があります。
>
> - 同じデータ、同じ分け方（訓練・検証）
> - 同じステップ数、同じバッチの大きさ、同じ学習率、同じ正則化（ドロップアウトの有無など）
> - **パラメータの数をなるべくそろえる**（大きくすれば良くなるのは当たり前なので）
> - **シード（乱数の種）を変えて何回か**やる。1回だけだと、たまたまかもしれない（第20章）
>
> ここでは、シード0と1の2回ずつ実験します。本当は5回以上やりたいところですが、時間との相談です。研究の世界では、この「何回やって、どれくらいばらついたか」を必ず書きます。

#### 📝 確認問題（`ch25-q01` · Lv.2 · 公平な比較実験の4条件（条件統一・パラメータ数・複数シード））

ベースライン（GPT-2型）と現代版ミニGPTを比べて「新しい部品が本当に効いたか」を確かめる実験で、**守るべき公平な比較の作法**として正しいものはどれですか？

- **a.** 現代版モデルだけパラメータ数を3倍・学習ステップ数を2倍へ増やし、最も検証損失が良かったシード1回だけを報告する
- **b.** データ分割・ステップ数・学習率・正則化をそろえ、パラメータ数も近づけたうえで複数シードで差とばらつきを比べる
- **c.** データ量を増やすため検証データ（`val_data`）を訓練データへ混ぜてから学習し、その後の検証損失を2モデルで比べる
- **d.** ベースラインには過学習防止のドロップアウトを残し、現代版にはドロップアウトを入れずにそのまま検証損失を比較する

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「変えたい部品以外の条件をすべて揃える（統制）」と「乱数による偶然の揺れを測る」の2本柱を思い出しましょう。
- **ヒント 2**：学習条件・正則化・パラメータ数の統一に加えて、1回の実行結果だけで判断しないための複数シード検証に注目してください。
- **正解**：`b`

**考え方**：比較実験の基本は「変えたい対象（独立変数）以外をすべて固定する（統制）」ことと、「乱数シードによる偶然の揺れを測る」ことです。

1. データ・ステップ数・バッチサイズ・学習率・ドロップアウト有無を統一する。
1. 総パラメータ数を近づける（SwiGLUで隠れ層を `4d` ではなく `8/3 d` にする理由もここ！）。
1. 複数シードで回し、平均の差がシード間のばらつきを上回るか確認する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/25-modern.html#ex-ch25-q01

</details>

## 部品1：RMSNorm

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
print(RMSNorm(4)(x))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
tensor([[0.3651, 0.7303, 1.0954, 1.4606]], grad_fn=<MulBackward0>)
```

</details>

#### 📐 数学のポイント：RMSNorm：9ステップで LayerNorm との違いを理解する

**① 解決したい具体的な問題**：深いTransformerでベクトルの大きさが暴走しないよう正規化したいが、毎層で何度も走る正規化をもっと軽く速くしたい。

**② LayerNorm のどこが重かったのか？**：第10章のLayerNormは「①平均 $\mu$ を求めて全要素から引く（中心化）→ ②分散を求めて標準偏差で割る（スケーリング）→ ③重み $w$ を掛けてバイアス $b$ を足す」という2段階の集計が必要でした。

**③ 新しい発想**：「大事なのはベクトルの**大きさ（スケール）を揃えること**であり、平均を0にずらす処理とバイアスは省いても性能はほぼ変わらないのでは？」→ **平均引きを丸ごと省略した RMSNorm**（Root Mean Square Normalization）！

**④ 記号の1つ1つの読み方**：
$$\mathrm{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}} \cdot w$$
- $d$：ベクトルの次元数（`dim`）
- $\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2}$：**RMS（2乗平均平方根）**。各要素を2乗 → 平均 → 平方根
- $\epsilon$：全要素0のときのゼロ除算を防ぐ極小値（例：`1e-6`）
- $w$：次元ごとの学習可能な倍率（初期値はすべて `1.0`）

**⑤ 小さな数字で手計算（LayerNorm との比較）**：$x = [1, 2, 3, 4]$、$w = [1, 1, 1, 1]$ のとき：
- 2乗：$[1^2, 2^2, 3^2, 4^2] = [1, 4, 9, 16]$
- 2乗の平均：$(1 + 4 + 9 + 16) / 4 = 30 / 4 = 7.5$
- RMS：$\sqrt{7.5} \approx 2.7386$
- 割り算：$[1/2.739,\ 2/2.739,\ 3/2.739,\ 4/2.739] \approx [0.365,\ 0.730,\ 1.095,\ 1.461]$（上のPyTorch出力と完全一致！）
- **LayerNorm との違い**：LayerNormなら先に平均 `2.5` を引くため `[-1.34, -0.45, +0.45, +1.34]`（平均0）になりますが、RMSNormは平均を引かないため全要素プラスのまま「2乗平均が1」に揃います！

**⑥ 純粋Pythonでの実装イメージ**：
```python no-run
import math
xs = [1.0, 2.0, 3.0, 4.0]
rms = math.sqrt(sum(v * v for v in xs) / len(xs) + 1e-6)
print([round(v / rms, 4) for v in xs])   # [0.3651, 0.7303, 1.0954, 1.4606]
```

**⑦ なぜ `torch.rsqrt` を使うのか**：`rsqrt(v)` は平方根の逆数 $1 / \sqrt{v}$ です。GPUでは割り算より逆数平方根の掛け算のほうが高速だからです。

**⑧ 検算の性質**：$w = 1$ のとき、RMSNormの出力 $y$ の2乗平均 $\frac{1}{d}\sum y_i^2$ は必ず $1.0$ になります。

**⑨ 現代LLMでの採用**：Llama・Qwen・Gemmaなど、近年の主要オープンLLMのほぼすべてがLayerNormの代わりにRMSNormを採用しています。

#### 📝 確認問題（`ch25-q02` · Lv.2 · RMSNormがLayerNormから省略した処理とその利点）

LlamaやQwenで採用されている **RMSNorm** は、第10章の **LayerNorm** と比べて何を省略していますか？

- **a.** ゼロ除算を防ぐための微小定数 $\epsilon$ を完全に省略し、入力が全ゼロでもそのまま逆数を計算する設計にした
- **b.** 学習可能なスケーリング重み $w$ をすべて廃止し、代わりに推論のたびに毎回ランダムな定数を掛ける仕組みにした
- **c.** 平均 $\mu$ を計算して引く操作（中心化）とバイアス項を省き、2乗平均の平方根で割って重み $w$ を掛けるだけにした
- **d.** 平方根や割り算によるスケール正規化をすべて省いてしまい、入力ベクトルに学習可能なバイアス項だけを足す形にした

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：LayerNormの式 $\frac{x - \mu}{\sqrt{\text{Var}(x)+\epsilon}}w + b$ と、RMSNormの式 $\frac{x}{\sqrt{\frac{1}{d}\sum x_i^2 + \epsilon}}w$ を見比べましょう。
- **ヒント 2**：RMSNormでは「大きさ（スケール）を揃える割り算」と「学習重み $w$」は残しつつ、中心化（平均引き）とバイアスが消えている点に注目してください。
- **正解**：`c`

**考え方**：大規模実験により「LayerNormの効能のほとんどは『平均を0にすること』ではなく『ベクトルの大きさ（スケール）を揃えること』にある」と分かったため、平均計算とバイアスを省いた高速なRMSNormが現代の標準になりました。

1. LayerNorm：$\dfrac{x - \mu}{\sqrt{\text{Var}(x) + \epsilon}} \cdot w + b$（平均 $\mu$ と分散の両方が必要）。
1. RMSNorm：$\dfrac{x}{\sqrt{\frac{1}{d}\sum x_i^2 + \epsilon}} \cdot w$（平均引きとバイアス $b$ を省略）。
1. 計算が少し速くシンプルになり、大規模LLMでの性能はほぼ同等。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/25-modern.html#ex-ch25-q02

</details>

#### 📝 確認問題（`ch25-q03` · Lv.3 · RMSNormのRMS値と出力ベクトルの手計算）

ベクトル $x = [1, 2, 3, 4]$、重み $w = [1, 1, 1, 1]$（$\epsilon \approx 0$）のとき、RMSNormの分母になる **RMS（2乗平均平方根）** $\sqrt{\frac{1}{4}(1^2 + 2^2 + 3^2 + 4^2)}$ はおよそいくつですか？（小数第2位まで、誤差 $\pm 0.05$ まで正解）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：RMS（2乗平均平方根）は「各要素を2乗する → 要素数で割って平均をとる → 平方根をとる」の3ステップで求めます。
- **ヒント 2**：まず $1^2 + 2^2 + 3^2 + 4^2$ の合計を4で割り、その値の平方根を小数第2位まで計算しましょう。
- **正解**：`2.74`

**考え方**：RMS $\approx 2.739$ で `[1, 2, 3, 4]` をそれぞれ割ると `[0.365, 0.730, 1.095, 1.461]` となり、本文のPyTorch実行出力とピタリ一致します！

1. 各要素を2乗：$1^2=1, 2^2=4, 3^2=9, 4^2=16$。
1. 平均をとる：$(1 + 4 + 9 + 16) / 4 = 30 / 4 = 7.5$。
1. 平方根をとる：$\sqrt{7.5} \approx 2.739$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/25-modern.html#ex-ch25-q03

</details>

*(この節の追加問題 `ch25-q04` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/25-modern.html で自動採点・復習できます)*

## 部品2：SwiGLU

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden, bias=False)   # 門番（SiLUを通す。Qwenの gate_proj）
        self.w3 = nn.Linear(dim, hidden, bias=False)   # 中身（Qwenの up_proj）
        self.w2 = nn.Linear(hidden, dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

#### 📐 数学のポイント：SwiGLU：9ステップで「門番（ゲート）」の計算と $\frac{8}{3}d$ ルールを理解する

**① 解決したい具体的な問題**：第12章のフィードフォワード層（FFN）`w2(GELU(w1(x)))` の表現力を、パラメータ数を増やさずに引き上げたい。

**② 従来のFFNの限界**：1本の全結合層で広げて活性化関数を通すだけでは、「どの特徴量を通し、どの特徴量を閉じるか」を別の経路で動的に制御できませんでした。

**③ 新しい発想**：広げる全結合層を**2本（中身 `w3` と門番 `w1`）**に分け、門番の出力を **SiLU**（$\mathrm{SiLU}(z) = z \cdot \sigma(z)$、GELUに似た滑らかな関数）に通してから、中身と**要素ごとに掛け算（$\odot$）**する！

**④ 数式とQwen/Llamaの層名の対応**：
$$\mathrm{SwiGLU}(x) = W_2\Big(\mathrm{SiLU}(W_1 x) \odot (W_3 x)\Big)$$
- $W_1$（`gate_proj`）：門番を作る線形層
- $W_3$（`up_proj`）：情報の中身を作る線形層
- $\odot$：同じ位置どうしの**要素ごとの掛け算**（内積ではない！）
- $W_2$（`down_proj`）：元の次元 `dim` に戻す線形層

**⑤ 小さな数字でゲートの動きを手計算**：
あるトークンで、門番 $\mathrm{SiLU}(W_1 x) = [0.05,\ 2.00]$、中身 $W_3 x = [10.0,\ 3.0]$ だったとします。
$$\mathrm{SiLU}(W_1 x) \odot (W_3 x) = [0.05 \times 10.0,\ 2.00 \times 3.0] = [0.5,\ 6.0]$$
チャネル0は中身が `10.0` と大きくても門番がほぼ閉じて `0.5` に抑えられ、チャネル1は門番が開いて `6.0` へ増幅されました！

**⑥ なぜ隠れ幅を $4d$ ではなく $\frac{8}{3}d$（≒2.67倍）にするのか？**
- 従来FFN（行列2つ、幅 $4d$）：パラメータ数 $= 2 \times d \times (4d) = 8d^2$
- SwiGLU（行列3つ、幅 $h$）：パラメータ数 $= 3 \times d \times h$
- これを等しくすると $3dh = 8d^2 \implies h = \dfrac{8}{3}d$！ 例：$d = 60$ なら $h = 160$（従来は $240$）でパラメータ数が完全一致します。

**⑦ Pythonでの最小イメージ**：
```python no-run
# 従来FFN：hidden = 4 * d（2行列：d->4d, 4d->d）
# SwiGLU：hidden = int(d * 8 / 3)（3行列：gate_proj, up_proj, down_proj）
```

**⑧ バイアスを省く理由**：現代LLMでは線形層の `bias=False` が定番で、RMSNormと合わせてシンプルかつ高速です。

**⑨ 第19章LoRAとのつながり**：第19章で `target_modules="all-linear"` を指定したとき、LoRAが付せんを貼った `gate_proj / up_proj / down_proj` の正体が、まさにこの3行の `SwiGLU` です！

#### 📝 確認問題（`ch25-q05` · Lv.2 · SwiGLUの「門番（ゲート）」の仕組みと2つの線形層の役割）

SwiGLUの式 `self.w2(F.silu(self.w1(x)) * self.w3(x))` で、`F.silu(self.w1(x))` と `self.w3(x)` の要素ごとの掛け算は何の役割を果たしていますか？

- **a.** 2つの隠れベクトルの内積を計算して、トークン全体に対する1個のスカラー確率（0〜1）へ圧縮する役割
- **b.** Self-Attention層のQueryとKeyの類似度計算を丸ごと置き換えて、Attention層自体を不要にする役割
- **c.** 入力シーケンスの語順を時間軸に沿って完全に逆転させ、未来から過去へ情報を流すための順序変換の役割
- **d.** 門番（ゲート）として通過率を決め、中身 `w3(x)` のどの情報をどれだけ通すか動的に開閉する役割

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：SwiGLUの「GLU」は Gated Linear Unit（ゲート付き線形層）の略です。門番（ゲート）が中身に掛け算されるイメージを思い浮かべましょう。
- **ヒント 2**：要素ごとの掛け算で、`F.silu(w1(x))` が `0` に近いチャネルと大きな正の値のチャネルとで、中身 `w3(x)` がどう開閉されるか考えます。
- **正解**：`d`

**考え方**：従来のFFNは1本の線形変換に活性化関数を通すだけでしたが、SwiGLUは「情報を作る層（`up_proj`＝`w3`）」と「どの情報を通すか決める門番（`gate_proj`＝`w1`）」を分けることで表現力を高めています。

1. 中身：`u = w3(x)`（`up_proj`）が候補となる特徴量を作る。
1. 門番：`g = SiLU(w1(x))`（`gate_proj`）が各特徴量を通す度合いを決める。
1. 掛け算：`g * u` で不要なチャネルを閉じ、必要なチャネルだけを通して `w2`（`down_proj`）で元の次元に戻す。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/25-modern.html#ex-ch25-q05

</details>

#### 📝 確認問題（`ch25-q06` · Lv.3 · SwiGLUの隠れ幅を 4d ではなく 8/3 d にするパラメータ数の計算）

従来のFFN（行列2つ、バイアスなし）は幅 $d$ を $4d$ に広げるため、パラメータ数は $2 \times d \times (4d) = 8d^2$ です。SwiGLUは行列が**3つ**あるため、同じパラメータ数 $8d^2$ にそろえるには隠れ幅を $\frac{8}{3}d$ にします。モデル幅 $d = 60$ のとき、SwiGLUの隠れ幅 $\frac{8}{3}d$ はいくつになりますか？（整数で回答）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：SwiGLUは線形層が3つあるため、従来FFN（2行列・幅 $4d$）とパラメータ数を一致させるには隠れ幅を $\frac{8}{3}d$ に設定します。
- **ヒント 2**：$d = 60$ に $\frac{8}{3}$ を掛けます。先に $60 \div 3$ を求めてから $8$ を掛けるとすっきり計算できます。
- **正解**：`160`

**考え方**：従来FFN：$2 \times 60 \times 240 = 28,800$ 個。SwiGLU：$3 \times 60 \times 160 = 28,800$ 個。隠れ幅を $\frac{8}{3}d$ にすることで、ちょうど同じパラメータ数で公平に比較できます！

1. 従来FFN（2行列、幅 $4d = 240$）：$2 \times 60 \times 240 = 28,800$。
1. SwiGLU（3行列、幅 $h$）：$3 \times 60 \times h = 28,800$。
1. $h = \dfrac{8}{3} \times 60 = 160$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/25-modern.html#ex-ch25-q06

</details>

## 部品3：RoPE（回転位置埋め込み）

第11章で、Attentionは順番を知らないので、**位置の埋め込み**を足しました。でも、足す方式には弱点があります。「3番目と5番目」と「103番目と105番目」は、どちらも「2つ離れている」のに、モデルにとっては別の組み合わせに見えてしまうのです。言葉の関係で大事なのは、多くの場合「**何文字離れているか**（相対位置）」なのに。

**RoPE**（Rotary Position Embedding）は、位置を**足す**代わりに、QueryとKeyのベクトルを位置に応じた角度だけ**回転**させます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def rope_cache(T, hs, base=10000.0):
    freqs = 1.0 / (base ** (torch.arange(0, hs, 2).float() / hs))   # ペアごとの回転の速さ
    angles = torch.outer(torch.arange(T).float(), freqs)            # [T, hs/2]：位置 × 速さ
    return torch.cos(angles), torch.sin(angles)

def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]            # 数を2つずつペアにする
    T = x.shape[-2]
    c, s = cos[:T], sin[:T]
    out = torch.stack([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1)   # 各ペアを回転
    return out.flatten(-2)

> **🐍 Python・PyTorchのポイント：1つ飛ばしのスライス、...、stack と flatten**
>
> 小さな例で見ると分かりやすいです。`a = torch.arange(6)`（中身は `[0, 1, 2, 3, 4, 5]`）とすると、
>
> - `a[0::2]` は「0番目から、2つ飛ばしで」→ `[0, 2, 4]`。`a[1::2]` は「1番目から、2つ飛ばしで」→ `[1, 3, 5]`。これで、隣どうしの数がペア（0と1、2と3、4と5）に分かれます
> - `x[..., 0::2]` の `...` は「前の軸は全部そのまま」という意味です。`x` の形が `[T, hs]` でも `[B, T, hs]` でも、**最後の軸だけ**を1つ飛ばしで取り出せます
> - `torch.stack([A, B], dim=-1)` は、AとBを最後の軸に並べて、ペアの形 `[…, 3, 2]` にします。`.flatten(-2)` は、最後の2つの軸をつないで1列に戻します。結果、`[A0, B0, A1, B1, A2, B2]` と、元の並び順に戻ります
> - `torch.outer(a, b)` は、`a` と `b` の全部の組み合わせの掛け算の表（九九の表のようなもの）を作ります
> - 下のセルの `q.expand(200, hs)` は、形 `[hs]` の `q` を、200行ぶん並べた `[200, hs]` に見せかけます（コピーはしません）。「200か所のどこにでも置ける」ようにするためです

#### 📐 数学のポイント：RoPE（回転位置埋め込み）：9ステップで2D回転と相対位置 $m - n$ を完全に理解する

**① 解決したい具体的な問題**：Attentionが「文頭から何文字目か（絶対位置）」ではなく、「**2つの単語が何文字離れているか（相対位置 $m - n$）**」を自然に感じ取れるようにし、学習時より長い文章にも拡張しやすくしたい。

**② 足し算式の位置埋め込み（`pos_emb`）の弱点**：位置表を足す方式では、学習した最大長 `block_size` より先の位置番号が存在せず、また位置 $(5, 3)$ と $(105, 103)$ が同じ「2つ隣」であることが式の上で表現できません。

**③ 新しい発想**：ベクトルの要素を隣どうし2つずつ $(x_1, x_2)$ のペア（2次元平面の座標）に分け、位置 $m$ のQueryを角度 $m\theta$、位置 $n$ のKeyを角度 $n\theta$ だけ**2D回転**させる！

**④ 2次元回転行列の式と読み方**：
$$\begin{pmatrix} x_1' \\ x_2' \end{pmatrix} = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}\begin{pmatrix} x_1 \\ x_2 \end{pmatrix} = \begin{pmatrix} x_1\cos\theta - x_2\sin\theta \\ x_1\sin\theta + x_2\cos\theta \end{pmatrix}$$
- $\theta$：回転角度（ラジアン）。RoPEではペアごとに異なる回転速度（周波数 `freqs`：速く回る秒針〜ゆっくり回る時針）を掛けた $m\theta$ を使う
- $\cos\theta, \sin\theta$：原点中心・半径1の円周上の横（x）と縦（y）の座標

**⑤ 具体的な90°回転の手計算**：
点 $(1, 0)$（右向きの矢印）を反時計回りに **90°**（$\theta = \pi/2$、$\cos(\pi/2)=0, \sin(\pi/2)=1$）回してみましょう！
$$x_1' = 1 \cdot 0 - 0 \cdot 1 = 0, \qquad x_2' = 1 \cdot 1 + 0 \cdot 0 = 1 \implies (1, 0) \to (0, 1)$$
見事に真上の矢印 $(0, 1)$ になりました！もう90°回すと $(0, 1) \to (-1, 0)$。そして長さは常に $\sqrt{x_1'^2 + x_2'^2} = 1$ のまま変わりません。

**⑥ なぜ内積に相対距離 $m - n$ だけが自然に残るのか？**：
2本の時計の針 $q, k$ を思い浮かべてください。ベクトルの内積は「**2本の長さの積 × なす角の $\cos$**」で決まります。
- 位置 $m$ のQueryを $m\theta$ 回し、位置 $n$ のKeyを $n\theta$ 回すと、両方の長さが変わらず、**2本の間の開き角の変化はちょうど $(m - n)\theta$ だけ**になります！
- 数式でも、$(R_{m\theta} q)^\top (R_{n\theta} k) = q^\top R_{(n - m)\theta} k$ となり、絶対位置 $m, n$ は消えて**相対位置 $m - n$ だけ**が残ります。

**⑦ 純粋Pythonでの最小コード**：
```python no-run
import math
def rotate_2d(x, y, theta):
    c, s = math.cos(theta), math.sin(theta)
    return x * c - y * s, x * s + y * c
print(tuple(round(v, 3) for v in rotate_2d(1.0, 0.0, math.pi / 2)))   # (0.0, 1.0)
```

**⑧ 実装上の必須条件**：2要素ペアを作るため `head_size`（`dim // n_head`）は必ず**偶数**である必要があります。また回転するのは Query と Key だけで、Value には適用しません。

**⑨ 実際のPyTorchコードで相対位置不変性を検算しよう**：下のセルで $(5, 3)$ と $(105, 103)$ が小数第4位まで完全一致することを確かめます！

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
hs = 16
cos, sin = rope_cache(200, hs)
torch.manual_seed(0)
q, k = torch.randn(hs), torch.randn(hs)

def score_at(m, n):
    qm = apply_rope(q.expand(200, hs), cos, sin)[m]   # 位置mに置いたQuery
    kn = apply_rope(k.expand(200, hs), cos, sin)[n]   # 位置nに置いたKey
    return (qm @ kn).item()

print(f"位置(5, 3)の点数：{score_at(5, 3):.4f}")
print(f"位置(105, 103)の点数：{score_at(105, 103):.4f}")
print(f"位置(5, 1)の点数：{score_at(5, 1):.4f}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
位置(5, 3)の点数：-1.8956
位置(105, 103)の点数：-1.8956
位置(5, 1)の点数：-3.3877
```

</details>

位置が(5, 3)でも(105, 103)でも、**離れ具合が同じなら同じ点数**です。一方、(5, 1)のように離れ具合が違えば点数も変わります。RoPEは、Attentionに「何文字離れているか」を自然に教えてくれるのです。足し算式の位置埋め込みのように「位置ごとの表」を持たないので、学習時より長い文章に伸ばす工夫（**RoPEの拡張**）もしやすく、長いコンテキストのLLMで重宝されています。

#### 📝 確認問題（`ch25-q07` · Lv.2 · RoPEの内積が相対位置 m-n で決まる性質と head_size 偶数の条件）

RoPE（回転位置埋め込み）を使って位置 $m$ のQueryと位置 $n$ のKeyを回転させたとき、2つの内積（Attentionの生の点数）は何によって決まりますか？

- **a.** QueryとKeyの角度は一切影響せず、Valueベクトル $V$ に掛けた回転角度だけでスコアが決まる
- **b.** 2つのトークンの位置番号を足し合わせた和 $m + n$ の大きさだけで一意に決まり、差にはよらない
- **c.** QueryとKeyの中身と、2つの位置の差（相対位置 $m - n$）だけで決まり、絶対位置にはよらない
- **d.** 回転によって位置情報を完全に打ち消す仕組みのため、どの位置ペアでも内積スコアは常に `0` になる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：時計の針が2本あるとき、両方の針を同じ角度だけ一緒に回しても「2本の針の間の開き角」はどうなるか想像しましょう。
- **ヒント 2**：位置 $m$ と $n$ にそれぞれ $m\theta, n\theta$ の回転を与えたとき、内積に残るのは絶対位置ではなく2つの位置の「差」である点に注目してください。
- **正解**：`c`

**考え方**：位置 $m$ と $n$ にそれぞれ $m\theta, n\theta$ の回転を与えると、内積はなす角の差 $(m - n)\theta$ だけに依存します。また固定長の位置テーブル `pos_emb` を持たないため、長文への拡張もしやすい利点があります。

1. 回転行列はベクトルの長さを変えず、角度だけを変える。
1. 位置 $m$ のQueryを $m\theta$、位置 $n$ のKeyを $n\theta$ 回すと、両者の相対角の変化はちょうど $(m - n)\theta$。
1. よって $(5, 3)$ と $(105, 103)$ はどちらも $m - n = 2$ なので完全に同じ内積スコアになる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/25-modern.html#ex-ch25-q07

</details>

#### 📝 確認問題（`ch25-q08` · Lv.3 · 2D回転行列による90°回転 (1, 0) → (0, 1) の手計算とコード予測）

RoPEの1ペアの回転式 $(x_1 \cos\theta - x_2 \sin\theta,\ x_1 \sin\theta + x_2 \cos\theta)$ で、点 $(1, 0)$ を反時計回りに **90°**（$\theta = \pi / 2$ ラジアン）回転させます。次のコードを実行すると何が表示されますか？

```python
import math

x1, x2 = 1.0, 0.0
theta = math.pi / 2
r1 = x1 * math.cos(theta) - x2 * math.sin(theta)
r2 = x1 * math.sin(theta) + x2 * math.cos(theta)
print(round(r1), round(r2))
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：2次元平面で右向きの矢印 $(1, 0)$ を反時計回りに90°回転させるとどちらの向きになるか図でイメージしましょう。
- **ヒント 2**：90°（$\pi/2$）では $\cos(\pi/2) = 0,\ \sin(\pi/2) = 1$ です。これを $r_1, r_2$ の式へ代入して2つの整数をスペース区切りで求めます。
- **期待される出力**：`0 1`

**考え方**：右向きの矢印 $(1, 0)$ を反時計回りに90°回すと真上向きの矢印 $(0, 1)$ になります。式でも $1\cdot 0 - 0\cdot 1 = 0$、$1\cdot 1 + 0\cdot 0 = 1$ と完全に一致します！

1. $\cos(\pi/2) = 0$、$\sin(\pi/2) = 1$。
1. $r_1 = 1\cdot 0 - 0\cdot 1 = 0$。
1. $r_2 = 1\cdot 1 + 0\cdot 0 = 1$ → 出力 `0 1`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/25-modern.html#ex-ch25-q08

</details>

*(この節の追加問題 `ch25-q09` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/25-modern.html で自動採点・復習できます)*

## 現代版ミニGPTを組み立てる

3つの部品を使って、GPTを組み立て直します。Attentionは、第15〜16章で学んだ「QKVをまとめて作り、`scaled_dot_product_attention` で計算する」書き方にします。位置の埋め込み（`pos_emb`）はなくなり、代わりにRoPEを使います。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
class ModernAttention(nn.Module):
    def __init__(self, dim, n_head, block_size):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        cos, sin = rope_cache(block_size, dim // n_head)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v)]
        q, k = apply_rope(q, self.cos, self.sin), apply_rope(k, self.cos, self.sin)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))

class ModernBlock(nn.Module):
    def __init__(self, dim, n_head, block_size):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(dim), RMSNorm(dim)
        self.attn = ModernAttention(dim, n_head, block_size)
        self.ffn = SwiGLU(dim, hidden=int(dim * 8 / 3))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.ffn(self.norm2(x))

class ModernGPT(nn.Module):
    def __init__(self, V, block_size, dim=64, n_head=4, n_layer=2):
        super().__init__()
        self.tok_emb = nn.Embedding(V, dim)
        self.blocks = nn.Sequential(*[ModernBlock(dim, n_head, block_size) for _ in range(n_layer)])
        self.norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, V, bias=False)
        for p in self.parameters():
            if p.dim() == 2:
                nn.init.normal_(p, std=0.02)

    def forward(self, idx, targets=None):
        x = self.blocks(self.tok_emb(idx))
        logits = self.lm_head(self.norm(x))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

modern = lambda: ModernGPT(V, block_size=T)
results["現代版（RMSNorm＋SwiGLU＋RoPE）"] = [experiment(modern, seed) for seed in [0, 1]]
for name, rs in results.items():
    losses = [r[0] for r in rs]
    print(f"{name}：検証損失 {[round(l, 3) for l in losses]}  平均 {sum(losses) / len(losses):.3f}  パラメータ {rs[0][1]:,}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ベースライン（GPT-2型）：検証損失 [3.21, 3.275]  平均 3.243  パラメータ 369,677
現代版（RMSNorm＋SwiGLU＋RoPE）：検証損失 [3.284, 3.229]  平均 3.256  パラメータ 362,176
```

</details>

**ボンゴ先生**：さて、結果を読もう。大事なのは、良くなったか悪くなったかより、**その差が、シードによるばらつきより大きいか**だ。

- 2つのモデルの平均の差：3.256 − 3.243 ＝ **0.013**
- 同じモデルでもシードを変えただけで：ベースラインは3.21〜3.275（幅 **0.065**）、現代版は3.229〜3.284（幅 **0.055**）

差は、ばらつきよりずっと小さい。だから結論は「**この規模の実験では、現代版の部品の効果は確認できなかった**」。がっかりするかもしれないが、これが誠実な結論だ。そして、それも立派な研究成果なんだよ。

**ミナト**：実は、教科書を作るときに最初にこの実験をしたら、ベースラインにだけドロップアウトが入っていて、ベースラインが3.10で圧勝していたそうですね。

**ボンゴ先生**：そうなんだ。まさに「公平な比較の作法」を破っていた。気づいて条件をそろえたら、差は消えた。逆に言うと、**この規模では、部品の違いよりもドロップアウト（第10章）のほうがずっと効く**、という発見でもある。データが少ないときは過学習対策が一番大事、という第13章の教訓と同じだね。研究では、こういう「失敗から見つかる発見」がよくある。だから、うまくいかなかった実験も記録に残しておくんだ。

> **💡 理論のポイント：小さな実験と大きな実験**
>
> 最新の部品の多くは、**大きなモデル・大量のデータ**で効果が確かめられたものです。2層・幅64・1500ステップという小さな実験では、効果がはっきり出ないこともありますし、逆に小さい規模でしか効かない工夫もあります。
>
> だから研究者は、小さな実験で素早くアイデアを試し、**有望なものだけを大きな実験で確かめる**、という順番で進めます。第15章のスケーリング則の話を思い出してください。小さな実験の結果を、どこまで大きな規模に当てはめてよいかを考えることも、研究の大事な仕事です。

#### 📝 確認問題（`ch25-q10` · Lv.3 · モデルの平均差とシード間のばらつきを比べて誠実に結論づける）

新しい部品を入れたモデルの検証損失が、シード1回だけの実験でベースラインより `0.01` 良くなりました。「この部品には効果がある」と結論づけてよいですか？

- **a.** 訓練データに対する損失がベースラインより順調に下がっているなら、検証損失のシード間ばらつきは一切無視してよい
- **b.** まだ言えない。複数シードで実験し平均差がシードのばらつきを上回るか、かつパラメータ数や条件が揃っているか確かめる必要がある
- **c.** 検証損失は乱数に左右されない絶対指標なので、シード1回でも `0.001` 以上良くなれば100%効果があると断言して問題ない
- **d.** 複数シードを回すと計算時間が無駄に延びるため、実験回数は増やさず理論的な直感だけで効果ありと結論づけるべきである

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：科学実験の鉄則である「モデル間の平均差（シグナル）が、乱数シードによる揺れ幅（ノイズ）より大きいか？」を思い出しましょう。
- **ヒント 2**：本文の実験ではモデル平均差 `0.013` に対して同一モデルのシードばらつき幅が `0.055〜0.065` あった点と、条件統制の確認に注目してください。
- **正解**：`b`

**考え方**：「差（シグナル）が、ばらつき（ノイズ）より大きいか？」を確認するのが科学実験の鉄則です。確認できなかったときに「この規模では効果は確認できなかった」と書くことが誠実な研究報告です。

1. 本文の結果：ベースライン `[3.210, 3.275]`（幅 0.065）、現代版 `[3.284, 3.229]`（幅 0.055）。
1. 両者の平均差はわずか `0.013` で、シードの揺れ幅よりずっと小さい。
1. 結論：「この規模の実験では、現代版部品の優位性は確認できなかった」。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/25-modern.html#ex-ch25-q10

</details>

#### 📝 確認問題（`ch25-q11` · Lv.3 · アブレーション（1部品ずつの切り分け実験）の目的）

RMSNorm・SwiGLU・RoPEの3つを一度に入れたモデルを評価したあと、「どの部品がどれだけ効いているか」を突き止めるために行う **アブレーション（ablation）実験** のやり方として正しいものはどれですか？

- **a.** 学習済みモデルの重みパラメータをすべてゼロへ削除してから、訓練を行わず初期状態のまま各部品の検証損失だけを比較する
- **b.** 3つの部品の入れ替えと同時に、学習率・データセット・層数・ステップ数も毎回ランダムに変えて1回ずつ実行し結果を比べる
- **c.** 他の条件を固定したまま、部品を1つだけ元の方式へ戻す比較をそれぞれの部品で行う
- **d.** 自分でコードを実行して計測するのではなく、先行論文の要旨に書かれた数値だけを抜き出して比較表にまとめる手法である

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：英語の「ablation（切除・除去）」が、複数の要素を同時に試したあとに各要素の真の貢献度を測る手法であることを思い出しましょう。
- **ヒント 2**：他の条件を固定したまま、3つの部品を1つずつ外したり戻したりして影響を個別に切り分けるやり方を選びます。
- **正解**：`c`

**考え方**：複数の改良を同時に導入すると、実は「Aは+0.05効いていたが、Bが-0.06足を引っ張っていた」という相殺を見逃します。1つずつ切り分けるアブレーションで各部品の真の貢献が見えます。

1. ベースライン（全部旧式）または全部入り（現代版）を基準にする。
1. 1部品だけ差し替えた変種（-RMSNorm, -SwiGLU, -RoPE）を同じシード群で実行する。
1. 各部品ごとの損失変化と速度変化を表にまとめる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/25-modern.html#ex-ch25-q11

</details>

## MoE：専門家を使い分ける

最近の大きなLLMの多くが採用している **MoE**（Mixture of Experts、専門家の混合）も、小さく作ってみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
class TinyMoE(nn.Module):
    def __init__(self, dim, n_experts=4, hidden=128):
        super().__init__()
        self.router = nn.Linear(dim, n_experts, bias=False)       # どの専門家に任せるか決める係
        self.experts = nn.ModuleList([SwiGLU(dim, hidden) for _ in range(n_experts)])

    def forward(self, x):
        B, T, C = x.shape
        flat = x.view(-1, C)
        gate = F.softmax(self.router(flat), dim=-1)                # 各トークンの、専門家ごとの重み
        weight, choice = gate.max(dim=-1)                          # 一番の専門家を1人だけ選ぶ（top-1）
        out = torch.zeros_like(flat)
        for e, expert in enumerate(self.experts):
            mask = choice == e
            if mask.any():
                out[mask] = weight[mask, None] * expert(flat[mask])
        self.last_choice = choice
        return out.view(B, T, C)

torch.manual_seed(0)
moe = TinyMoE(64)
x = torch.randn(2, 16, 64)
y = moe(x)
print(y.shape)
print("各専門家が担当したトークン数：", torch.bincount(moe.last_choice, minlength=4).tolist())

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
torch.Size([2, 16, 64])
各専門家が担当したトークン数： [6, 6, 14, 6]
```

</details>

#### 📐 数学のポイント：MoE（Mixture of Experts）：9ステップで「総パラメータ vs 活性パラメータ」と負荷分散を理解する

**① 解決したい具体的な問題**：モデルの知識容量（パラメータ数）を大きく増やしたいが、1トークンを生成するごとの計算量（FLOPs）は増やしたくない！

**② 普通の密（Dense）モデルの壁**：普通のTransformerではパラメータを10倍にすると、1トークンあたりの掛け算回数もそのまま10倍に跳ね上がり、推論が遅く高額になります。

**③ 新しい発想**：各ブロックのFFN（SwiGLU）を複数個（専門家 $E_1, \dots, E_N$）並べ、軽量な**ルーター**がトークンごとに上位 $k$ 人（**top-$k$**）の専門家だけを選んで計算する！

**④ 数式と記号の読み方**：
$$g = \mathrm{softmax}(W_{\text{router}} x), \qquad y = \sum_{e \in \text{Top-}k(g)} g_e \cdot E_e(x)$$
- $W_{\text{router}}$：各専門家の適性スコアを出す小さな線形層
- $g_e$：選ばれた専門家 $e$ の重み（softmax確率）
- $\text{Top-}k(g)$：スコア上位 $k$ 人（例：top-1 や top-2）だけの集合。選ばれなかった専門家は**計算を完全にスキップ**！

**⑤ 具体的な数字で「総パラメータ vs 活性パラメータ」を手計算**：
共通層（埋め込み・Attention等）が **20億**、専門家 **8人**（1人 **10億**）、**top-2** ルーティングのとき：
- **総パラメータ数（知識容量・必要なVRAM）** $= 20 + 8 \times 10 = \mathbf{100\text{億}}$
- **1トークンあたりの活性パラメータ数（計算量・速さ）** $= 20 + 2 \times 10 = \mathbf{40\text{億}}$
つまり **100億パラメータ級の知識を持ちながら、40億パラメータ級の軽さで推論**できます！（ただし全100億の重みはVRAMに待機させる必要がある点に注意！）

**⑥ 負荷の偏り（ルーティング崩壊）と負荷分散損失**：
普通に学習すると「たまたま最初によく呼ばれた1〜2人の専門家だけが急速に賢くなり、ルーターが全員分をその専門家へ送り続けて他の専門家が死んでしまう」現象が起きます。上の出力でも `[6, 6, 14, 6]` と偏りが起きていますね。これを防ぐため、各専門家の担当割合が均等からズレるほどペナルティを足す**負荷分散の補助損失（load balancing loss）**を加えます。

**⑦ Pythonでの最小イメージ**：
```python no-run
# 1トークンあたりの計算コスト ＝ 共通層 ＋ top_k × 専門家1人分
total_params  = 20 + 8 * 10   # 100億（メモリを支配）
active_params = 20 + 2 * 10   #  40億（推論速度を支配）
```

**⑧ トレードオフの整理**：「計算は軽いが、メモリ（VRAM）は総パラメータ分しっかり必要」。病院に各科の専門医が全員待機していて、患者ごとに担当医1〜2人だけが診察するイメージです。

**⑨ 最新モデルでの活躍**：Mixtral・DeepSeek・Qwen3-MoE など、近年の最先端オープンLLMの多くがこのMoE構造を採用しています！

#### 📝 確認問題（`ch25-q12` · Lv.3 · MoEの「総パラメータ数」と「1トークンあたりの活性パラメータ数」の計算）

あるMoEモデルでは、全トークンが共通で通る層（埋め込み・Attention・出力ヘッド等）が **20億パラメータ**、MoEの専門家（FFN）が **8人** いて、専門家1人あたり **10億パラメータ** あります。ルーターが各トークンごとに **top-2**（上位2人の専門家だけ）を選ぶとき、①総パラメータ数は 100億 ですが、**1トークンの推論で実際に計算に使われる活性パラメータ数**は何億パラメータですか？（整数で回答）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：1トークンの推論で実際に動く「活性パラメータ数」は、全トークン共通の層と選ばれた top-k 人の専門家の合計です。
- **ヒント 2**：共通層（20億）に、top-2 で選ばれた専門家2人分（$2 \times 10$億）を足して合計を求めましょう。
- **正解**：`40` 億

**考え方**：総パラメータ数は $20 + 8 \times 10 = 100$億（大容量の知識を記憶できる）なのに、1トークンの計算量は $20 + 2 \times 10 = 40$億分（軽くて速い！）で済みます。これがMoEの「大きいのに軽い」秘密です。

1. 共通層（全トークンで必ず実行）：20億。
1. 専門家8人のうち top-2 のみ実行：$2 \times 10 = 20$億。
1. 活性パラメータ数：$20 + 20 = 40$億（総パラメータ100億の40%の計算量で動く！）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/25-modern.html#ex-ch25-q12

</details>

#### 📝 確認問題（`ch25-q13` · Lv.3 · MoEのルーターと負荷分散損失（Load Balancing Loss）が必要な理由）

MoE（Mixture of Experts）をそのまま学習させると「ルーターが特定の1〜2人の専門家ばかりに全トークンを送り、残りの専門家がほとんど使われない」という現象が起きがちです。この問題と一般的な対策はどれですか？

- **a.** 数値オーバーフロー。専門家の総数を8人から1人だけへ削減してしまえば、MoEの利点と性能が最も高く発揮される
- **b.** データリーク。トークナイザーの語彙サイズを半分へ減らして入力トークンの種類を抑えれば、自動的に偏りが解消される
- **c.** 勾配消失問題。ルーターの `softmax` を削除して全トークンを常に0番目の専門家へ固定転送すれば安定して学習できる
- **d.** 負荷の偏り（ルーティング崩壊）。各専門家の担当トークン数が均等に近づくよう罰則を与える負荷分散の補助損失を足す

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「たまたま最初に多く選ばれた専門家ほど成長が速く、ルーターにさらに選ばれやすくなる」現象と、それを均す工夫を思い出しましょう。
- **ヒント 2**：専門家の担当割合が偏るほどペナルティが増える補助損失（load balancing loss）を学習損失に加える仕組みに注目してください。
- **正解**：`d`

**考え方**：せっかく複数の専門家を用意しても1人しか使われなければ、メモリだけ食う普通の小モデルになってしまいます。負荷分散損失（またはルーターバイアス調整）で全専門家にバランスよく仕事を振ることがMoE成功の鍵です。

1. 問題：たまたま多く選ばれた専門家が早く上達 → ルーターがさらにその専門家ばかり選ぶ → 他の専門家が育たない（死ぬ）。
1. 結果：並列計算のボトルネックになり、せっかくの総パラメータ容量も活かせない。
1. 対策：各専門家の担当割合が偏るほどペナルティが増える負荷分散損失（load balancing loss）を加える。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/25-modern.html#ex-ch25-q13

</details>

## 最新研究の地図

ここまでで、あなたは現代のLLMの主要な部品を、すべて自分の手で書いたことになります。最後に、この先の研究の世界を、地図として眺めておきましょう。どれも、この教科書で学んだことの延長線上にあります。

> **💡 理論のポイント：長いコンテキスト**
>
> 何十万〜何百万トークンを一度に読むLLMが登場しています。課題は、Attentionの計算とKVキャッシュのメモリが長さとともに増えること（第16章）。RoPEの拡張、GQA、FlashAttention、一部のトークンだけに注目する**疎なAttention**などを組み合わせて実現しています。

#### 💡 理論のポイント：推論モデルと「考える時間」

答える前に長く考える**推論モデル**は、推論のとき（テスト時）に多くの計算（トークン）を使うほど賢くなる、という性質を利用しています（**テスト時の計算のスケーリング**）。第22章で触れた「答えが確かめられる問題での強化学習（RLVR）」で、考え方そのものを学ばせます。

GRPOの中心にあるのは、驚くほど単純な計算です。同じ問題に何通りも答えさせ、それぞれの報酬が**グループの平均よりどれだけ良いか**（アドバンテージ）で、その答えを強めたり弱めたりします。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
rewards = torch.tensor([1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0])   # 8通りの答えの正誤
advantage = (rewards - rewards.mean()) / (rewards.std() + 1e-6)
print("グループの平均：", rewards.mean().item())
print("アドバンテージ：", [round(a, 2) for a in advantage.tolist()])

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
グループの平均： 0.375
アドバンテージ： [1.21, -0.72, -0.72, 1.21, 1.21, -0.72, -0.72, -0.72]
```

</details>

正解した答えはプラス（強める）、間違えた答えはマイナス（弱める）。平均と比べるので、「全員正解」の簡単な問題や「全員不正解」の難しすぎる問題からは、何も学びません（アドバンテージが0になる）。ちょうどいい難しさの問題から学ぶ、よくできた仕組みです。

> **💡 理論のポイント：Transformer以外の選択肢**
>
> Attentionの計算量が長さの2乗で増えるのを避けるため、**状態空間モデル**（SSM、代表例はMamba）のように、RNN（第9章のMLPを時間方向にくり返し使うような仕組み）の考え方を現代的に作り直したモデルも研究されています。過去の情報を、決まった大きさの「状態」に圧縮して持ち歩くので、長さに比例した計算で済みます。Transformerと組み合わせた**ハイブリッド**のモデルも登場しています。

> **💡 理論のポイント：マルチモーダル**
>
> 画像を小さな正方形（パッチ）に切り分け、各パッチを1つの「トークン」としてベクトルにすれば、Transformerは文章と同じように画像も扱えます（Vision Transformer）。画像のトークンと文章のトークンを並べて入力すれば、画像について会話するLLMになります。音声や動画も同じ発想です。第2章の「何でもトークンにする」が、ここまで広がるのです。

> **💡 理論のポイント：解釈可能性**
>
> LLMの中で何が起きているのかを調べる研究を **解釈可能性**（interpretability）と言います。第13章でヘッドの注目の仕方を可視化したのは、その入口です。特定の概念に反応するニューロンや、ある処理を担う回路を見つけたり、中間の層のベクトルから「何を考えているか」を読み取ったり（プロービング）します。安全なAIを作るためにも、とても重要な分野です。

> **💡 理論のポイント：蒸留**
>
> 大きなモデル（先生）の出力の確率を、小さなモデル（生徒）にまねさせて学習させる方法を **蒸留**（distillation）と言います。正解の1トークンだけでなく、先生の「2位、3位の候補への確率」からも学べるので、小さなモデルが効率よく賢くなります。第3章の「確率の分布」そのものが、教材になるわけです。

> **🔬 やってみよう：自分で部品を1つずつ試す**
>
> 3つの部品を一度に入れると、どれが効いたのか分かりません。研究では、1つずつ入れたり外したりして、効果を切り分けます（**アブレーション**）。`ModernBlock` の `RMSNorm` を `nn.LayerNorm` に戻す、`SwiGLU` を第12章の `FeedForward` に戻す、などして、どの部品がどれくらい効いているか調べてみましょう。結果を表にまとめれば、立派な研究ノートです。

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **RoPEで形のエラー**：`head_size`（`dim // n_head`）が**偶数**でないと、2つずつのペアが作れません。
> - **現代版のほうが損失が悪い**：小さな規模や短い学習では、そういうこともあります。学習率など、ハイパーパラメータは部品ごとに最適な値が違うこともあるので、それぞれで少し調整するのが本当は公平です。
> - **MoEで一部の専門家が全然使われない**：負荷の偏りです。ルーターの出力に、偏りを罰する損失を足すのが一般的な対策です。

#### 📝 確認問題（`ch25-q14` · Lv.3 · GRPOのグループ平均とアドバンテージの計算・学習信号の仕組み）

GRPOで同じ数学問題に8通りの答えを生成させたところ、正誤の報酬が `[1, 0, 0, 1, 1, 0, 0, 0]`（8問中3問正解）でした。このグループの平均報酬 `rewards.mean()` はいくつですか？（小数で回答）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：グループ平均報酬 `rewards.mean()` は、8通りの報酬をすべて合計して個数8で割った値です。
- **ヒント 2**：報酬 `1` が3個、`0` が5個なので、合計 `3` を総数 `8` で割って小数に直しましょう。
- **正解**：`0.375`

**考え方**：平均報酬は `3 / 8 = 0.375` なので、正解（`1.0`）は平均より高くプラスのアドバンテージ（強化）、不正解（`0.0`）は平均より低くマイナスのアドバンテージ（抑制）になります。

1. 正解の合計：$1 + 1 + 1 = 3$。
1. 平均：$3 \div 8 = 0.375$。
1. 正解の偏差：$1.0 - 0.375 = +0.625$（標準偏差で割って `+1.21`）、不正解は `-0.375`（`-0.72`）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/25-modern.html#ex-ch25-q14

</details>

#### 📝 確認問題（`ch25-q15` · Lv.5 · 【総合】最新研究の地図（Attention vs SSM）と部品アブレーション計画）

① TransformerのAttentionと状態空間モデル（SSM：Mambaなど）の計算量・記憶の持ち方の違いを説明してください。② さらに、第25章の `ModernGPT`（RMSNorm＋SwiGLU＋RoPE）について「3つのうちどの部品が最も検証損失と推論速度に影響するか」を公平に調べるアブレーション実験の設計（比較する4〜5条件・統一すべき変数・シード数・判定基準）を具体的に書いてください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：①ではAttentionとSSMの「長さ $T$ に対する計算量」と「過去文脈の記憶方式」を対比し、②ではアブレーションの比較条件と統制ルールを整理しましょう。
- **ヒント 2**：①は $O(T^2)$＋全履歴参照 vs $O(T)$＋固定サイズ状態圧縮、②は1部品ずつ変える4〜5条件・統一変数（パラメータ数・正則化等）・複数シードのばらつき判定基準を具体化します。
- **解答の目安**：① TransformerのAttentionは過去の全トークンを直接参照するため文脈検索精度が非常に高い一方、計算量が長さの2乗 $O(T^2)$・KVキャッシュも長さに比例して増大します。対してSSM（Mambaなど）は過去の情報を固定サイズの状態ベクトルに圧縮して持ち歩くため、計算量が長さ $T$ に比例する $O(T)$ で超長文に強い反面、遠い過去の細かい1文字をそのまま取り出す能力ではAttentionに劣る傾向があり、近年は両者のハイブリッドも研究されています。② アブレーション計画では、比較条件として (1) ベースライン（LayerNorm＋通常FFN＋絶対位置）、(2) +RMSNormのみ、(3) +SwiGLUのみ（隠れ幅 $8/3 d$ で総パラメータ一致）、(4) +RoPEのみ、(5) 全部入り（ModernGPT）の5条件を用意します。データ分割・ステップ数・バッチサイズ・学習率・ドロップアウト有無（0.0）と総パラメータ数をすべて統一し、各条件を複数シード（例：5シード `0..4`）で学習します。検証損失の平均・標準偏差（または信頼区間）および1ステップあたりの計算時間を計測し、平均の差がシード間のばらつきを上回った部品だけ「この規模で有効」と誠実に結論づけます。

**考え方**：最新アーキテクチャのトレードオフを理解し、自分の仮説をアブレーションで検証できる力こそが「論文を読んで試せる研究者」の第一歩です。

1. Attention（$O(T^2)$、過去の全KVを保持し精密参照）vs SSM（$O(T)$、固定状態に圧縮し長文高速）。
1. 5条件（ベースライン＋1部品ずつの3条件＋全部入り）で切り分ける。
1. パラメータ数・正則化・ステップ数を統一し、複数シードで平均とばらつきを比較する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/25-modern.html#ex-ch25-q15

</details>

**✅ できたらチェック**

- [ ] RMSNorm・SwiGLU・RoPEをそれぞれ自分で実装した
- [ ] RoPEで、点数が相対位置だけで決まることを確かめた
- [ ] 現代版ミニGPTを組み立て、ベースラインと公平に比べた
- [ ] 差がシードのばらつきより大きいかを考えて、結論を書いた
- [ ] MoEのルーティングと負荷の偏りを観察した
- [ ] GRPOのアドバンテージを計算した

> **📝 まとめ**
>
> - 現代のLLMは、**RMSNorm**・**SwiGLU**・**RoPE**など、GPT-2から少しずつ改良された部品を使う
> - **RoPE**は、QueryとKeyを位置に応じて回転させ、点数を**相対位置**で決まるようにする
> - 比較実験は、**部品以外をそろえ**、**複数のシード**でばらつきと差を比べる
> - **MoE**は、ルーターがトークンを一部の専門家に振り分ける。大きいのに計算は軽い
> - 研究の地図：長いコンテキスト・推論モデル（RLVR・GRPO）・SSM・マルチモーダル・解釈可能性・蒸留

**ボンゴ先生**：キミはもう、論文を読んで「それ、自分のGPTに入れて試せるな」と思える人になった。次はいよいよ最終章。卒業制作のテーマを決めて、2030年までの地図を描こう。ここまで来たキミなら、どこへでも行ける。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/25-modern.html
- **次の章**：[第26章 卒業制作と、2030年への地図](https://sakunokyumei.github.io/llm-research-textbook/26-capstone.html)